# Chapter 5 MyElite fine-tuning with Qwen

- 데이터셋: `/data/chapter5/loyalty_qa_train.jsonl`, `/data/chapter5/loyalty_qa_val.jsonl`
- 모델: `Qwen/Qwen2.5-0.5B-Instruct`
- 산출물: `/model-assets`


In [1]:
import importlib.util
import subprocess
import sys

package_modules = {
  "transformers": "transformers",
  "datasets": "datasets",
  "peft": "peft",
  "accelerate": "accelerate",
  "sentencepiece": "sentencepiece",
  "protobuf": "google.protobuf",
}

missing_packages = [
  package
  for package, module in package_modules.items()
  if importlib.util.find_spec(module) is None
]

if missing_packages:
  subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

print("missing packages installed:", missing_packages or "none")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 11.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 9.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 11.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 8.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 10.1 MB/s  0:00:00
  Attempting uninstall: dill━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/16 [regex]
    Found existing installation: dill 0.4.0━━━━━━━━━━━━━━━━━━━  4/16 [regex]
    Uninstalling dill-0.4.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/16 [regex]
      Successfully uninstalled dill-0.4.0━━━━━━━━━━━━━━━━━━━━━  4/16 [regex]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [peft]2m

In [2]:
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TRAIN_DATASET_FILE = "/data/chapter5/loyalty_qa_train.jsonl"
EVAL_DATASET_FILE = "/data/chapter5/loyalty_qa_val.jsonl"
MODEL_OUTPUT_DIR = Path("/model-assets")
MAX_STEPS = 20
MAX_LENGTH = 512

print("cuda available:", torch.cuda.is_available())
print("train dataset:", TRAIN_DATASET_FILE)
print("eval dataset:", EVAL_DATASET_FILE)


cuda available: True
train dataset: /data/chapter5/loyalty_qa_train.jsonl
eval dataset: /data/chapter5/loyalty_qa_val.jsonl


- 원본 Qwen 모델에 LoRA adapter를 붙이는 단계
- 학습이 시작된 건 아니고, “학습할 작은 파라미터만 추가하고 나머지는 고정”하는 준비단계

## 코드의미

```python
model = get_peft_model(model, config)
```

- peft가 원본 Qwen 모델을 감쌉니다.
- LoraConfig에 적은 target module에 LoRA layer를 추가합니다.
- 여기서는 이런 모듈에 붙입니다.
- ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] attention과 MLP 쪽 주요 projection layer에 LoRA adapter를 붙입니다.

```python
model.print_trainable_parameters()
```

- 전체 모델 파라미터 중 실제로 학습되는 파라미터 수를 출력합니다.
- LoRA fine-tuning은 원본 모델 전체를 학습하지 않고 adapter만 학습합니다.
- 그래서 trainable parameter 비율이 작게 나오는 게 정상입니다

In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model_options = {"torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32, "trust_remote_code": True}
if torch.cuda.is_available():
  model_options["device_map"] = "auto"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_options)
model.config.use_cache = False
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
  model.enable_input_require_grads()

config = LoraConfig(
  r=16,
  lora_alpha=32,
  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
  bias="none",
  lora_dropout=0.05,
  task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [4]:
def format_training_text(example):
  messages = [
    {"role": "user", "content": example["prompt"]},
    {"role": "assistant", "content": example["response"]},
  ]
  return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def tokenize_dataset(dataset):
  def tokenize_example(example):
    tokens = tokenizer(format_training_text(example), truncation=True, max_length=MAX_LENGTH)
    return tokens

  return dataset.map(tokenize_example, remove_columns=dataset.column_names)


train_dataset = tokenize_dataset(load_dataset("json", data_files=TRAIN_DATASET_FILE, split="train"))
eval_dataset = tokenize_dataset(load_dataset("json", data_files=EVAL_DATASET_FILE, split="train"))
len(train_dataset), len(eval_dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

(96, 29)

In [5]:
def move_inputs_to_device(inputs, device):
  if isinstance(inputs, torch.Tensor):
    return inputs.to(device)
  return {key: value.to(device) for key, value in inputs.items()}


def generate_with_inputs(inputs, **kwargs):
  if isinstance(inputs, torch.Tensor):
    return model.generate(inputs, **kwargs)
  return model.generate(**inputs, **kwargs)


def get_input_length(inputs):
  if isinstance(inputs, torch.Tensor):
    return inputs.shape[-1]
  return inputs["input_ids"].shape[-1]


def generate_text(prompt):
  messages = [{"role": "user", "content": prompt}]
  inputs = move_inputs_to_device(
    tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"),
    model.device,
  )
  with torch.no_grad():
    outputs = generate_with_inputs(inputs, max_new_tokens=100, repetition_penalty=1.15)
  return tokenizer.decode(outputs[0][get_input_length(inputs):], skip_special_tokens=True)


eval_prompt = "[MyElite Loyalty Program FAQ]: What is the maximum cashback I can earn?"
print(generate_text(eval_prompt))


As an AI language model, I don't have access to specific details about your Elite loyalty program and its rules or policies. However, typically in most programs, you may be able to earn up to 10% of your total spending on eligible items per month for rewards such as cash back.

To get accurate information about what's possible with your particular program, please check their official website or contact customer support there directly. They should be able to provide detailed guidance based on your account balance and any


In [6]:
trainer = Trainer(
  model=model,
  train_dataset=train_dataset,
  eval_dataset=eval_dataset,
  args=TrainingArguments(
    output_dir="/tmp/qwen25-myelite-notebook",
    warmup_steps=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    max_steps=MAX_STEPS,
    learning_rate=2.5e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=10,
    do_eval=True,
    report_to="none",
  ),
  data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()


Step,Training Loss,Validation Loss
10,2.875156,2.559673
20,2.188085,2.166511


TrainOutput(global_step=20, training_loss=2.8106964588165284, metrics={'train_runtime': 6.1284, 'train_samples_per_second': 6.527, 'train_steps_per_second': 3.263, 'total_flos': 7608215247360.0, 'train_loss': 2.8106964588165284, 'epoch': 0.4166666666666667})

- 학습한 가중치를 저장(adapter_model.safetensors)

In [7]:
print(generate_text(eval_prompt))

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))
print("saved:", MODEL_OUTPUT_DIR)


Your Elite loyalty program does not have any specific limit on how much you can receive in cash back. However, we do recommend that you check your account regularly to ensure that there are no outstanding charges or fees associated with your purchases.
saved: /model-assets


In [10]:
!ls -alh $MODEL_OUTPUT_DIR

total 45M
drwxrwsrwx 2 root users 4.0K May 24 03:14 .
drwxr-xr-x 1 root root  4.0K May 24 08:23 ..
-rw-rw-r-- 1 root users 5.1K May 24 08:30 README.md
-rw-rw-r-- 1 root users 1.1K May 24 08:30 adapter_config.json
-rw-rw-r-- 1 root users  34M May 24 08:30 adapter_model.safetensors
-rw-rw-r-- 1 root users 2.5K May 24 08:30 chat_template.jinja
-rw-rw-r-- 1 root users  11M May 24 08:30 tokenizer.json
-rw-rw-r-- 1 root users  694 May 24 08:30 tokenizer_config.json
-rw-rw-r-- 1 root users 5.2K May 24 08:30 training_args.bin
